# 3. Qualidade dos Dados

Após a construção da camada Bronze, será realizada uma avaliação da qualidade dos dados históricos de produção offshore da ANP referentes ao período de **2005 a 2025**.

Esta etapa busca compreender as características dos dados e identificar possíveis inconsistências antes da aplicação de qualquer transformação.

Os resultados obtidos servirão como base para a definição das regras de tratamento e padronização que serão aplicadas posteriormente na construção da camada Silver.

## 3.1 Objetivo

O objetivo deste notebook é avaliar a qualidade dos dados armazenados nas **11 tabelas históricas da camada Bronze**, identificando características ou inconsistências que possam afetar sua posterior integração e utilização nas análises.

A avaliação também busca distinguir problemas que exigem tratamento de características dos dados que devem ser preservadas, evitando alterações sem justificativa.

Ao final do notebook, os principais achados serão consolidados e utilizados para definir as regras que orientarão a construção da camada Silver no Notebook 04.

## 3.2 Avaliação inicial da qualidade dos dados

A avaliação da qualidade será realizada sobre os dados armazenados na camada Bronze, antes da aplicação das transformações previstas para a camada Silver.

Essa avaliação tem como objetivo identificar características ou inconsistências que possam comprometer a integração dos diferentes períodos históricos ou as análises realizadas posteriormente.

Serão avaliadas as seguintes dimensões de qualidade:

- **Estrutura:** quantidade de colunas, nomenclatura e tipos de dados;
- **Completude:** presença de valores nulos nos atributos;
- **Unicidade:** ocorrência de registros duplicados;
- **Consistência:** comportamento dos atributos categóricos ao longo do histórico;
- **Validade:** presença de valores incompatíveis com a natureza dos atributos quantitativos.

As decisões de tratamento serão tomadas somente após essa avaliação, evitando alterações nos dados sem uma justificativa baseada nas características observadas.

### 3.2.1 Estrutura e tipos de dados

A primeira análise de qualidade consiste na comparação da estrutura das 11 tabelas da camada Bronze.

Embora a exploração dos arquivos de origem tenha demonstrado que todos possuem 21 atributos e a mesma estrutura lógica, foram identificadas diferenças na representação dos cabeçalhos entre alguns períodos. Além disso, como a inferência de schema foi realizada separadamente durante a ingestão, atributos equivalentes podem ter sido persistidos com tipos de dados diferentes entre as tabelas Bronze.

O objetivo desta análise é identificar automaticamente essas divergências antes da consolidação das tabelas, permitindo definir posteriormente as regras de padronização da camada Silver.

#### 3.2.1.1 Levantamento dos schemas

Inicialmente, são coletados os nomes das colunas e os respectivos tipos de dados das 11 tabelas Bronze.

Esse levantamento cria uma estrutura única de comparação dos schemas e servirá como base para a identificação automática das divergências existentes entre os períodos.

In [0]:
from pyspark.sql import Row

# Lista das tabelas da camada Bronze
tabelas_bronze = [
    "producao_mar_2005_2009",
    "producao_mar_2010_2012",
    "producao_mar_2013_2015",
    "producao_mar_2016_2018",
    "producao_mar_2019",
    "producao_mar_2020",
    "producao_mar_2021",
    "producao_mar_2022",
    "producao_mar_2023",
    "producao_mar_2024",
    "producao_mar_2025"
]

comparacao_schema = []

for tabela in tabelas_bronze:

    df = spark.table(f"workspace.bronze.{tabela}")

    for campo in df.schema.fields:

        comparacao_schema.append(
            Row(
                tabela=tabela,
                coluna=campo.name,
                tipo_dado=campo.dataType.simpleString()
            )
        )

df_schema = spark.createDataFrame(comparacao_schema)

#### 3.2.1.2 Normalização dos nomes para comparação

Durante a exploração dos arquivos de origem no notebook `01_apresentacao&exploracao_Dados`, foi identificado que os arquivos referentes aos anos de 2022 a 2025 apresentam os nomes das colunas delimitados por colchetes, enquanto os períodos anteriores não possuem esse padrão.

Para permitir a comparação dos schemas entre os diferentes períodos, será criada uma versão normalizada dos nomes das colunas exclusivamente para fins de análise, removendo os colchetes quando presentes.

Essa normalização não altera as tabelas da camada Bronze. A padronização definitiva dos nomes dos atributos será realizada posteriormente durante a construção da camada Silver.

In [0]:
from pyspark.sql import functions as F

# Normaliza temporariamente os nomes das colunas para permitir a comparação
# entre os períodos, removendo os colchetes presentes nos arquivos mais recentes.
# Essa transformação é aplicada apenas à estrutura de análise e não altera a camada Bronze.
df_schema_comparacao = (
    df_schema
    .withColumn(
        "coluna_comparacao",
        F.regexp_replace(F.col("coluna"), r"^\[|\]$", "")
    )
)

#### 3.2.1.3 Identificação das divergências de tipos

Após a normalização temporária dos nomes, é possível comparar os tipos de dados atribuídos a cada atributo ao longo do histórico.

Como atributos equivalentes devem possuir tipos compatíveis para serem posteriormente consolidados, serão exibidas apenas as colunas que apresentam **mais de um tipo de dado** entre as 11 tabelas Bronze.

Essa abordagem direciona a análise para as inconsistências efetivamente encontradas, evitando a inspeção manual de todos os atributos e períodos.

In [0]:
# Agrupa os atributos equivalentes e identifica somente aqueles que apresentam
# mais de um tipo de dado entre as tabelas históricas da camada Bronze.
colunas_divergentes = (
    df_schema_comparacao
    .groupBy("coluna_comparacao")
    .agg(
        F.countDistinct("tipo_dado").alias("quantidade_tipos"),
        F.sort_array(
            F.collect_set("tipo_dado")
        ).alias("tipos_encontrados")
    )
    .filter(F.col("quantidade_tipos") > 1)
    .orderBy("coluna_comparacao")
)

# Exibe apenas os atributos que apresentam divergência de tipos entre os períodos.
display(colunas_divergentes)

coluna_comparacao,quantidade_tipos,tipos_encontrados
Ano,2,"List(int, string)"
Injeção de Gás Carbônico (Mm³),2,"List(int, string)"
Injeção de Nitrogênio (Mm³),2,"List(int, string)"
Injeção de Outros Fluidos (m³),2,"List(int, string)"
Injeção de Polímeros (m³),2,"List(int, string)"
Injeção de Vapor de Água (t),2,"List(int, string)"
Injeção de Água para Descarte (m³),2,"List(int, string)"


#### 3.2.1.4 Localização das divergências de tipos

Após identificar quais atributos possuem tipos divergentes, a análise é detalhada para verificar em quais períodos cada representação ocorre.

Para tornar a comparação mais objetiva, as tabelas que apresentam o mesmo tipo de dado para cada atributo são agrupadas, permitindo visualizar a distribuição das divergências ao longo do histórico sem repetir individualmente todas as combinações entre atributo e período.

Essa análise fornece evidências para a definição dos tipos de dados que serão adotados posteriormente na camada Silver.

In [0]:
# Resume os períodos em que cada tipo de dado ocorre para os atributos com divergências.
detalhamento_divergencias = (
    df_schema_comparacao
    .join(
        colunas_divergentes.select("coluna_comparacao"),
        on="coluna_comparacao",
        how="inner"
    )
    .groupBy(
        "coluna_comparacao",
        "tipo_dado"
    )
    .agg(
        F.concat_ws(
            ", ",
            F.sort_array(F.collect_set("tabela"))
        ).alias("tabelas_periodos")
    )
    .orderBy(
        "coluna_comparacao",
        "tipo_dado"
    )
)

display(detalhamento_divergencias)

coluna_comparacao,tipo_dado,tabelas_periodos
Ano,int,"producao_mar_2005_2009, producao_mar_2010_2012, producao_mar_2013_2015, producao_mar_2016_2018, producao_mar_2022, producao_mar_2023, producao_mar_2024, producao_mar_2025"
Ano,string,"producao_mar_2019, producao_mar_2020, producao_mar_2021"
Injeção de Gás Carbônico (Mm³),int,"producao_mar_2005_2009, producao_mar_2010_2012"
Injeção de Gás Carbônico (Mm³),string,"producao_mar_2013_2015, producao_mar_2016_2018, producao_mar_2019, producao_mar_2020, producao_mar_2021, producao_mar_2022, producao_mar_2023, producao_mar_2024, producao_mar_2025"
Injeção de Nitrogênio (Mm³),int,"producao_mar_2005_2009, producao_mar_2010_2012, producao_mar_2013_2015, producao_mar_2016_2018, producao_mar_2022"
Injeção de Nitrogênio (Mm³),string,"producao_mar_2019, producao_mar_2020, producao_mar_2021, producao_mar_2023, producao_mar_2024, producao_mar_2025"
Injeção de Outros Fluidos (m³),int,"producao_mar_2010_2012, producao_mar_2013_2015, producao_mar_2016_2018, producao_mar_2022, producao_mar_2023, producao_mar_2024, producao_mar_2025"
Injeção de Outros Fluidos (m³),string,"producao_mar_2005_2009, producao_mar_2019, producao_mar_2020, producao_mar_2021"
Injeção de Polímeros (m³),int,"producao_mar_2010_2012, producao_mar_2013_2015, producao_mar_2022"
Injeção de Polímeros (m³),string,"producao_mar_2005_2009, producao_mar_2016_2018, producao_mar_2019, producao_mar_2020, producao_mar_2021, producao_mar_2023, producao_mar_2024, producao_mar_2025"


##### 3.2.1.4.1 Análise das divergências de tipos encontradas
A análise identificou **7 atributos com divergências de tipos de dados entre os períodos históricos**.

O atributo `Ano` apresenta variação entre `integer` e `string`. Além dele, seis atributos relacionados à injeção de fluidos também alternam entre esses dois tipos ao longo do histórico.

Nos períodos de **2019, 2020 e 2021**, a predominância de atributos do tipo `string` está relacionada à estratégia específica de ingestão adotada para tratar as particularidades de parsing identificadas nesses arquivos, conforme documentado no notebook `02_ingestao_bronze`. Nesses períodos, os atributos foram intencionalmente mantidos como `string` na camada Bronze, priorizando a correta interpretação e preservação dos dados. A padronização dos tipos será realizada posteriormente, de forma explícita, durante a construção da camada Silver.

As divergências observadas demonstram a necessidade de padronização explícita dos tipos de dados antes da consolidação do histórico.

#### 3.2.1.5 Resultado da análise estrutural

A comparação dos schemas das 11 tabelas Bronze confirmou a existência de diferenças de nomenclatura e de tipos de dados ao longo do histórico.

Foram identificados **sete atributos com divergência de tipo**, todos apresentando alternância entre `integer` e `string`:

- `Ano`;
- `Injeção de Gás Carbônico (Mm³)`;
- `Injeção de Nitrogênio (Mm³)`;
- `Injeção de Outros Fluidos (m³)`;
- `Injeção de Polímeros (m³)`;
- `Injeção de Vapor de Água (t)`;
- `Injeção de Água para Descarte (m³)`.

Nos arquivos referentes a **2019, 2020 e 2021**, os atributos foram persistidos como `string` em razão da estratégia específica de ingestão adotada para corrigir o padrão de formatação identificado nesses períodos. Conforme documentado no notebook `02_ingestao_bronze`, essa decisão preserva a correta interpretação dos registros na camada Bronze e transfere a definição explícita dos tipos de dados para a camada Silver.

Além disso, alguns atributos quantitativos apresentam alternância entre `integer` e `string` também em outros períodos, evidenciando diferenças decorrentes da inferência automática de tipos realizada durante a ingestão.

Em relação à nomenclatura, os arquivos referentes aos anos de **2022, 2023, 2024 e 2025** apresentam seus cabeçalhos delimitados por colchetes. Durante esta análise, esses nomes foram normalizados apenas temporariamente para permitir a comparação entre os schemas, sem modificar as tabelas Bronze.

Os resultados demonstram que, embora as tabelas representem a mesma estrutura lógica de **21 atributos**, existem diferenças técnicas que precisam ser tratadas antes da consolidação do histórico. Portanto, a construção da camada Silver deverá incluir a **padronização dos nomes das colunas e a definição explícita dos tipos de dados**, eliminando as diferenças existentes entre os períodos.

### 3.2.2 Análise de completude

A segunda dimensão de qualidade avaliada é a **completude dos dados**, por meio da identificação e quantificação de valores nulos nos atributos das tabelas Bronze.

A presença de valores nulos não será considerada automaticamente um problema de qualidade, pois determinados atributos podem não ser aplicáveis a todos os registros. Esse comportamento pode ocorrer, por exemplo, em variáveis associadas a diferentes tipos de produção ou injeção.

Nesta etapa, o objetivo é identificar quais atributos apresentam valores ausentes e qual a representatividade dessas ocorrências no histórico analisado.

Nenhuma substituição ou remoção de valores será realizada neste momento. Os resultados serão utilizados posteriormente para definir as regras de tratamento da camada Silver.

#### 3.2.2.1 Identificação dos valores nulos

Inicialmente, será contabilizada a ocorrência de valores nulos em cada atributo das 11 tabelas Bronze.

Os nomes das colunas serão normalizados apenas para fins de comparação entre os períodos, removendo os colchetes presentes nos cabeçalhos de 2022 a 2025. Essa operação não modifica as tabelas Bronze.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Row

resultado_nulos = []

for tabela in tabelas_bronze:

    df = spark.table(f"workspace.bronze.{tabela}")

    total_registros = df.count()

    # Calcula a quantidade de valores nulos de todas as colunas da tabela
    contagem_nulos = df.select([
        F.sum(
            F.when(F.col(f"`{coluna}`").isNull(), 1).otherwise(0)
        ).alias(coluna)
        for coluna in df.columns
    ]).collect()[0]

    # Organiza o resultado no formato: tabela, coluna, total e quantidade de nulos
    for coluna in df.columns:

        # Remove os colchetes apenas do nome utilizado na comparação
        coluna_comparacao = coluna.strip("[]")

        resultado_nulos.append(
            Row(
                tabela=tabela,
                coluna=coluna_comparacao,
                total_registros=total_registros,
                registros_nulos=contagem_nulos[coluna]
            )
        )

df_nulos = spark.createDataFrame(resultado_nulos)

#### 3.2.2.2 Consolidação da análise de completude

Para facilitar a interpretação dos resultados, as ocorrências de valores nulos serão consolidadas por atributo considerando todo o período analisado.

Será calculado o total de registros, a quantidade de valores nulos e o respectivo percentual de nulidade. Serão exibidos apenas os atributos que apresentarem pelo menos um valor nulo no histórico.

Essa abordagem permite concentrar a análise nos atributos que efetivamente apresentam ausência de informação.

In [0]:
resumo_nulos = (
    df_nulos
    .groupBy("coluna")
    .agg(
        F.sum("total_registros").alias("total_registros"),
        F.sum("registros_nulos").alias("registros_nulos")
    )
    .withColumn(
        "percentual_nulos",
        F.round(
            (F.col("registros_nulos") / F.col("total_registros")) * 100,
            2
        )
    )
    .filter(F.col("registros_nulos") > 0)
    .orderBy(F.desc("percentual_nulos"))
)

display(resumo_nulos)

coluna,total_registros,registros_nulos,percentual_nulos
Injeção de Gás (Mm³),323307,241248,74.62
Injeção de Água para Recuperação Secundária (m³),323307,241248,74.62
Injeção de Água para Descarte (m³),323307,241248,74.62
Injeção de Gás Carbônico (Mm³),323307,241248,74.62
Injeção de Nitrogênio (Mm³),323307,241248,74.62
Injeção de Vapor de Água (t),323307,241248,74.62
Injeção de Polímeros (m³),323307,241248,74.62
Injeção de Outros Fluidos (m³),323307,241248,74.62
Produção de Óleo (m³),323307,74957,23.18
Produção de Condensado (m³),323307,74957,23.18


#### 3.2.2.3 Distribuição dos valores nulos entre os períodos

Além da análise consolidada, será verificada a distribuição dos valores nulos entre as tabelas históricas.

Essa etapa permite identificar se a ausência de informação ocorre de forma recorrente ao longo do histórico ou se está concentrada em determinados conjuntos de dados.

Para tornar a análise mais objetiva, os resultados serão resumidos por atributo, indicando os períodos em que foram identificados valores nulos e os percentuais mínimo e máximo de ausência observados nesses períodos.

In [0]:
# Resume por atributo os períodos com ocorrência de valores nulos e a faixa percentual observada.
detalhamento_nulos = (
    df_nulos
    .filter(F.col("registros_nulos") > 0)
    .withColumn(
        "percentual_nulos",
        F.round(
            (F.col("registros_nulos") / F.col("total_registros")) * 100,
            2
        )
    )
    .groupBy("coluna")
    .agg(
        F.concat_ws(
            ", ",
            F.sort_array(F.collect_set("tabela"))
        ).alias("tabelas_com_nulos"),
        F.round(F.min("percentual_nulos"), 2).alias("percentual_minimo"),
        F.round(F.max("percentual_nulos"), 2).alias("percentual_maximo")
    )
    .orderBy("coluna")
)

display(detalhamento_nulos)

coluna,tabelas_com_nulos,percentual_minimo,percentual_maximo
Estado,"producao_mar_2022, producao_mar_2023",0.04,0.05
Injeção de Gás (Mm³),"producao_mar_2005_2009, producao_mar_2010_2012, producao_mar_2013_2015, producao_mar_2016_2018, producao_mar_2019, producao_mar_2020, producao_mar_2021, producao_mar_2022, producao_mar_2023, producao_mar_2024, producao_mar_2025",64.26,78.69
Injeção de Gás Carbônico (Mm³),"producao_mar_2005_2009, producao_mar_2010_2012, producao_mar_2013_2015, producao_mar_2016_2018, producao_mar_2019, producao_mar_2020, producao_mar_2021, producao_mar_2022, producao_mar_2023, producao_mar_2024, producao_mar_2025",64.26,78.69
Injeção de Nitrogênio (Mm³),"producao_mar_2005_2009, producao_mar_2010_2012, producao_mar_2013_2015, producao_mar_2016_2018, producao_mar_2019, producao_mar_2020, producao_mar_2021, producao_mar_2022, producao_mar_2023, producao_mar_2024, producao_mar_2025",64.26,78.69
Injeção de Outros Fluidos (m³),"producao_mar_2005_2009, producao_mar_2010_2012, producao_mar_2013_2015, producao_mar_2016_2018, producao_mar_2019, producao_mar_2020, producao_mar_2021, producao_mar_2022, producao_mar_2023, producao_mar_2024, producao_mar_2025",64.26,78.69
Injeção de Polímeros (m³),"producao_mar_2005_2009, producao_mar_2010_2012, producao_mar_2013_2015, producao_mar_2016_2018, producao_mar_2019, producao_mar_2020, producao_mar_2021, producao_mar_2022, producao_mar_2023, producao_mar_2024, producao_mar_2025",64.26,78.69
Injeção de Vapor de Água (t),"producao_mar_2005_2009, producao_mar_2010_2012, producao_mar_2013_2015, producao_mar_2016_2018, producao_mar_2019, producao_mar_2020, producao_mar_2021, producao_mar_2022, producao_mar_2023, producao_mar_2024, producao_mar_2025",64.26,78.69
Injeção de Água para Descarte (m³),"producao_mar_2005_2009, producao_mar_2010_2012, producao_mar_2013_2015, producao_mar_2016_2018, producao_mar_2019, producao_mar_2020, producao_mar_2021, producao_mar_2022, producao_mar_2023, producao_mar_2024, producao_mar_2025",64.26,78.69
Injeção de Água para Recuperação Secundária (m³),"producao_mar_2005_2009, producao_mar_2010_2012, producao_mar_2013_2015, producao_mar_2016_2018, producao_mar_2019, producao_mar_2020, producao_mar_2021, producao_mar_2022, producao_mar_2023, producao_mar_2024, producao_mar_2025",64.26,78.69
Instalação,producao_mar_2022,21.3,21.3


#### 3.2.2.4 Resultado da análise de completude

A análise de completude dos **323.307 registros** revelou diferentes padrões de ausência de informação entre os atributos.

Os oito atributos relacionados às operações de injeção apresentaram os maiores percentuais de nulidade, com **241.248 registros nulos (74,62%)** em cada atributo no histórico consolidado. O detalhamento por período mostrou que a ausência dessas informações ocorre de forma recorrente ao longo dos conjuntos históricos analisados.

Os cinco atributos relacionados à produção — óleo, condensado, gás associado, gás não associado e água — também apresentaram comportamento semelhante entre si, com **74.957 registros nulos (23,18%)** em cada atributo no histórico consolidado. A proporção varia entre os períodos, de **16,82%** no conjunto de 2005 a 2009 até aproximadamente **35%** nos períodos mais recentes.

Também foram identificados **4.095 valores nulos no atributo `Instalação`**, correspondentes a **1,27%** do histórico e concentrados exclusivamente nos dados de 2022. No atributo `Estado`, foram identificados apenas **16 registros nulos**, sendo 10 em 2022 e 6 em 2023.

Os resultados demonstram que a ausência de informação não ocorre de maneira uniforme entre os atributos e apresenta padrões específicos ao longo do histórico. Entretanto, a presença de valores nulos não será interpretada automaticamente como erro, especialmente nos atributos relacionados à produção e à injeção, cuja ausência pode estar associada às características dos registros de origem.

Nesta etapa, portanto, **nenhum valor nulo será substituído ou removido**. A definição das regras de tratamento na camada Silver considerará o significado dos atributos e sua utilização nas análises, evitando principalmente a substituição indiscriminada de valores ausentes por zero.

### 3.2.3 Análise de duplicidades

A dimensão de unicidade tem como objetivo identificar possíveis registros duplicados no conjunto de dados.

Entretanto, a identificação de duplicidades não deve se limitar à comparação de linhas completamente idênticas. Antes de definir qualquer regra de remoção, é necessário compreender a granularidade dos dados e verificar se registros aparentemente repetidos representam efetivamente duplicidades ou ocorrências válidas da fonte.

Nesta etapa serão realizadas duas verificações:

1. identificação de linhas completamente duplicadas dentro de cada tabela Bronze;
2. investigação da granularidade dos registros a partir dos atributos que representam o período e o poço.

Nenhum registro será removido durante esta análise. Os resultados serão utilizados posteriormente para definir as regras de tratamento da camada Silver.

#### 3.2.3.1 Verificação de linhas completamente duplicadas

Inicialmente, será verificada a existência de registros completamente idênticos dentro de cada tabela Bronze.

Essa análise compara todas as colunas de cada tabela e contabiliza quantos registros excedem a primeira ocorrência de uma mesma combinação de valores.

O objetivo é identificar possíveis repetições integrais dos registros sem realizar qualquer exclusão nesta etapa.

In [0]:
# Verifica a ocorrência de linhas completamente duplicadas em cada tabela Bronze.
resultado_duplicidades = []

for tabela in tabelas_bronze:

    df = spark.table(f"workspace.bronze.{tabela}")

    total_registros = df.count()
    total_distintos = df.distinct().count()

    registros_duplicados = total_registros - total_distintos

    resultado_duplicidades.append(
        (
            tabela,
            total_registros,
            total_distintos,
            registros_duplicados
        )
    )

df_duplicidades = spark.createDataFrame(
    resultado_duplicidades,
    [
        "tabela",
        "total_registros",
        "registros_distintos",
        "registros_duplicados"
    ]
)

display(df_duplicidades.orderBy("tabela"))

tabela,total_registros,registros_distintos,registros_duplicados
producao_mar_2005_2009,60004,60004,0
producao_mar_2010_2012,37103,37103,0
producao_mar_2013_2015,38758,38758,0
producao_mar_2016_2018,56337,56337,0
producao_mar_2019,22173,22173,0
producao_mar_2020,21815,21815,0
producao_mar_2021,21149,21149,0
producao_mar_2022,19225,19225,0
producao_mar_2023,14824,14824,0
producao_mar_2024,15229,15229,0


#### 3.2.3.2 Investigação da granularidade dos registros

A ausência de linhas completamente duplicadas não garante, por si só, a unicidade dos registros segundo a granularidade esperada da base.

Como os dados representam informações mensais de produção por poço, será investigada a combinação entre `Mês/Ano` e `Poço` para verificar se um mesmo poço apresenta mais de um registro no mesmo período.

Nesta etapa, a ocorrência de combinações repetidas não será considerada automaticamente uma duplicidade. Caso sejam identificadas, os demais atributos desses registros deverão ser analisados para determinar se representam ocorrências válidas ou inconsistências nos dados.

In [0]:
# Verifica a ocorrência de mais de um registro para a combinação
# entre poço e período em cada tabela da camada Bronze.
resultado_granularidade = []

for tabela in tabelas_bronze:

    df = spark.table(f"workspace.bronze.{tabela}")

    # Identifica os nomes das colunas considerando a diferença
    # de nomenclatura existente nos arquivos mais recentes.
    coluna_poco = "[Poço]" if "[Poço]" in df.columns else "Poço"
    coluna_periodo = "[Mês/Ano]" if "[Mês/Ano]" in df.columns else "Mês/Ano"

    combinacoes_repetidas = (
        df
        .groupBy(coluna_poco, coluna_periodo)
        .count()
        .filter(F.col("count") > 1)
    )

    quantidade_combinacoes = combinacoes_repetidas.count()

    resultado_granularidade.append(
        (
            tabela,
            quantidade_combinacoes
        )
    )

df_granularidade = spark.createDataFrame(
    resultado_granularidade,
    [
        "tabela",
        "combinacoes_poco_periodo_repetidas"
    ]
)

display(df_granularidade.orderBy("tabela"))

tabela,combinacoes_poco_periodo_repetidas
producao_mar_2005_2009,196
producao_mar_2010_2012,838
producao_mar_2013_2015,603
producao_mar_2016_2018,3690
producao_mar_2019,2765
producao_mar_2020,3177
producao_mar_2021,2797
producao_mar_2022,2740
producao_mar_2023,2876
producao_mar_2024,3246


#### 3.2.3.3 Refinamento da granularidade com o atributo Campo

A análise anterior permite verificar se a combinação entre `Poço` e `Mês/Ano` é suficiente para identificar unicamente os registros. Caso sejam encontradas múltiplas ocorrências, é necessário investigar quais outros atributos diferenciam essas linhas antes de classificá-las como duplicidades.

A inspeção inicial dos registros indicou que o atributo `Campo` pode representar uma dimensão adicional dessa granularidade. Dessa forma, será avaliada, em todas as 11 tabelas Bronze, a combinação entre `Mês/Ano`, `Poço` e `Campo`.

O objetivo é verificar se a inclusão do campo elimina as combinações repetidas ao longo de todo o período histórico, permitindo compreender de forma mais consistente a granularidade observada na fonte.

In [0]:
# Avalia em todos os períodos se a inclusão do Campo na combinação
# período + poço é suficiente para distinguir os registros.
resultado_granularidade_campo = []

for tabela in tabelas_bronze:

    df = spark.table(f"workspace.bronze.{tabela}")

    # Considera a diferença de nomenclatura dos arquivos de 2022 a 2025.
    coluna_poco = "[Poço]" if "[Poço]" in df.columns else "Poço"
    coluna_periodo = "[Mês/Ano]" if "[Mês/Ano]" in df.columns else "Mês/Ano"
    coluna_campo = "[Campo]" if "[Campo]" in df.columns else "Campo"

    combinacoes_repetidas = (
        df
        .groupBy(
            coluna_periodo,
            coluna_poco,
            coluna_campo
        )
        .count()
        .filter(F.col("count") > 1)
    )

    quantidade_combinacoes = combinacoes_repetidas.count()

    resultado_granularidade_campo.append(
        (
            tabela,
            quantidade_combinacoes
        )
    )

df_granularidade_campo = spark.createDataFrame(
    resultado_granularidade_campo,
    [
        "tabela",
        "combinacoes_periodo_poco_campo_repetidas"
    ]
)

display(df_granularidade_campo.orderBy("tabela"))

tabela,combinacoes_periodo_poco_campo_repetidas
producao_mar_2005_2009,152
producao_mar_2010_2012,598
producao_mar_2013_2015,269
producao_mar_2016_2018,32
producao_mar_2019,0
producao_mar_2020,0
producao_mar_2021,1
producao_mar_2022,2
producao_mar_2023,0
producao_mar_2024,0


#### 3.2.3.4 Refinamento da análise de granularidade

A inclusão do atributo `Campo` reduziu a quantidade de combinações repetidas, mas não foi suficiente para garantir a unicidade dos registros em todos os períodos históricos.

As repetições permaneceram principalmente nos conjuntos de dados mais antigos, além de ocorrências pontuais em 2021 e 2022.

Dessa forma, a granularidade será novamente investigada considerando o atributo `Instalação`, verificando se um mesmo poço pode apresentar, no mesmo período e campo, registros associados a diferentes instalações.

Será avaliada a combinação entre `Mês/Ano`, `Poço`, `Campo` e `Instalação` em todas as tabelas Bronze.

In [0]:
# Refina a investigação da granularidade incluindo a instalação
# associada ao registro.
resultado_granularidade_instalacao = []

for tabela in tabelas_bronze:

    df = spark.table(f"workspace.bronze.{tabela}")

    coluna_poco = "[Poço]" if "[Poço]" in df.columns else "Poço"
    coluna_periodo = "[Mês/Ano]" if "[Mês/Ano]" in df.columns else "Mês/Ano"
    coluna_campo = "[Campo]" if "[Campo]" in df.columns else "Campo"
    coluna_instalacao = "[Instalação]" if "[Instalação]" in df.columns else "Instalação"

    combinacoes_repetidas = (
        df
        .groupBy(
            coluna_periodo,
            coluna_poco,
            coluna_campo,
            coluna_instalacao
        )
        .count()
        .filter(F.col("count") > 1)
    )

    resultado_granularidade_instalacao.append(
        (
            tabela,
            combinacoes_repetidas.count()
        )
    )

df_granularidade_instalacao = spark.createDataFrame(
    resultado_granularidade_instalacao,
    [
        "tabela",
        "combinacoes_periodo_poco_campo_instalacao_repetidas"
    ]
)

display(df_granularidade_instalacao.orderBy("tabela"))

tabela,combinacoes_periodo_poco_campo_instalacao_repetidas
producao_mar_2005_2009,0
producao_mar_2010_2012,0
producao_mar_2013_2015,0
producao_mar_2016_2018,0
producao_mar_2019,0
producao_mar_2020,0
producao_mar_2021,0
producao_mar_2022,0
producao_mar_2023,0
producao_mar_2024,0


#### 3.2.3.5 Resultado da análise de duplicidades

A análise de unicidade não identificou linhas completamente duplicadas em nenhuma das 11 tabelas da camada Bronze. Em todos os períodos, a quantidade de registros distintos foi igual à quantidade total de registros.

Em seguida, foi investigada a granularidade dos dados a partir dos atributos associados ao período e à identificação do poço. A combinação entre `Mês/Ano` e `Poço` apresentou múltiplas ocorrências, demonstrando que esses dois atributos, isoladamente, não são suficientes para identificar unicamente os registros.

A inclusão do atributo `Campo` reduziu as ocorrências repetidas, mas ainda foram identificadas combinações não únicas em alguns períodos históricos. Esse resultado indicou a existência de um nível adicional de detalhamento nos registros.

Por fim, foi avaliada a combinação entre `Mês/Ano`, `Poço`, `Campo` e `Instalação`. Nessa configuração, **não foram identificadas combinações repetidas em nenhuma das 11 tabelas Bronze**.

Os resultados indicam, portanto, que a granularidade observada nos dados corresponde a **um registro por combinação de período, poço, campo e instalação**.

Dessa forma, as repetições encontradas nas análises intermediárias não caracterizam duplicidades, mas refletem diferentes associações de campo e instalação para um mesmo poço ao longo de determinado período.

Como não foram identificadas linhas completamente duplicadas nem repetições na granularidade observada, **nenhum registro será removido por duplicidade durante a construção da camada Silver**.

### 3.2.4 Consistência dos atributos categóricos

A análise de consistência tem como objetivo verificar se os principais atributos categóricos apresentam representações compatíveis com seu significado e comportamento esperado ao longo do histórico.

Nesta etapa serão avaliados os atributos `Estado`, `Bacia`, `Campo`, `Ambiente` e `Instalação`, relevantes para a caracterização geográfica e operacional dos registros.

A quantidade de valores distintos será utilizada como indicador inicial para identificar comportamentos atípicos entre os períodos. Variações de cardinalidade não serão consideradas automaticamente inconsistências, uma vez que estados, campos e instalações presentes nos registros podem variar ao longo do histórico.

Durante a primeira execução desta análise foi identificada uma inconsistência nos dados de 2019 a 2021, que levou à revisão da estratégia de ingestão da camada Bronze. Por esse motivo, esta seção também documenta o diagnóstico realizado e a posterior validação da correção.

#### 3.2.4.1 Identificação inicial da inconsistência

Na primeira execução da análise de cardinalidade dos atributos categóricos, realizada antes da correção da ingestão, os períodos de **2019, 2020 e 2021** apresentaram um comportamento significativamente diferente dos demais.

Foram observadas quantidades anormalmente elevadas de valores distintos em atributos como `Estado`, `Bacia`, `Campo` e `Ambiente`, incompatíveis com a natureza dessas informações.

A figura abaixo registra o resultado obtido nessa etapa inicial da avaliação de qualidade.

**Figura 1 — Resultado inicial da análise de cardinalidade dos atributos categóricos**

> *![Evidência da validação](https://github.com/renatanegreiross/Sprint2_EngenhariadeDados_MVP/blob/d24fe89e4b31d67b811a3774740172b67371cfc9/doc/evidencias/validacao_qualidade_3_2_4_1.png)*

A identificação dessas cardinalidades anômalas indicou que parte dos registros poderia ter sido interpretada incorretamente durante a ingestão.

Diante desse resultado, a construção da camada Silver foi temporariamente interrompida e a investigação retornou aos arquivos de origem e à lógica de ingestão da camada Bronze, buscando identificar a causa da inconsistência antes da aplicação de qualquer transformação.

#### 3.2.4.2 Diagnóstico da inconsistência

Após a identificação das cardinalidades anômalas nos dados de 2019, 2020 e 2021, foi necessário investigar se a inconsistência estava presente nos dados originais ou se havia sido introduzida durante o processo de ingestão.

Para isso, a investigação retornou aos arquivos armazenados na camada RAW, preservados sem transformações, permitindo analisar sua estrutura original e testar a hipótese de interpretação incorreta durante o parsing dos arquivos CSV.

##### 3.2.4.2.1 Investigação dos arquivos de origem

Inicialmente, os arquivos originais foram analisados diretamente como texto, evitando a interpretação automática de sua estrutura como CSV.

A inspeção revelou que parte dos registros dos arquivos de **2019, 2020 e 2021** apresentava um padrão de formatação diferente dos demais períodos: algumas linhas estavam delimitadas externamente por aspas e os campos que já utilizavam aspas internamente apresentavam esses caracteres duplicados.

Esse comportamento levantou a hipótese de que o parser CSV poderia interpretar incorretamente a estrutura dessas linhas, provocando o deslocamento dos valores entre as colunas observado durante a análise inicial dos atributos categóricos.

##### 3.2.4.2.2 Identificação do padrão de aspas externas

Para verificar a extensão do padrão identificado, será contabilizada, em cada arquivo de origem, a quantidade de registros cuja linha se inicia com aspas.

A leitura será realizada diretamente como texto, preservando a representação original dos registros e evitando qualquer interpretação prévia da estrutura CSV.

Essa análise permitirá verificar em quais períodos o padrão ocorre e quantos registros são afetados.

In [0]:
# Quantifica o padrão de formatação que originou a inconsistência
# identificada durante a avaliação inicial da camada Bronze.

from pyspark.sql import functions as F
from pyspark.sql import Row

caminho_raw = "/Volumes/workspace/default/anp_raw/"

arquivos_raw = [
    "producao-mar-2005-2009.csv",
    "producao-mar-2010-2012.csv",
    "producao-mar-2013-2015.csv",
    "producao-mar-2016-2018.csv",
    "producao-mar-2019.csv",
    "producao-mar-2020.csv",
    "producao-mar-2021.csv",
    "producao-mar-2022.csv",
    "producao-mar-2023.csv",
    "producao_por_poco_2024.csv",
    "producao-mar-2025.csv"
]

resultado_aspas = []

for arquivo in arquivos_raw:

    caminho = f"{caminho_raw}{arquivo}"

    # A leitura como texto preserva a representação original da linha,
    # permitindo diagnosticar o problema antes da interpretação como CSV.
    df_texto = spark.read.text(caminho)

    total_linhas = df_texto.count() - 1

    linhas_com_aspas = (
        df_texto
        .filter(F.col("value").startswith('"'))
        .count()
    )

    resultado_aspas.append(
        Row(
            arquivo=arquivo,
            total_registros=total_linhas,
            linhas_iniciadas_com_aspas=linhas_com_aspas
        )
    )

df_diagnostico_aspas = spark.createDataFrame(resultado_aspas)

display(df_diagnostico_aspas)

arquivo,total_registros,linhas_iniciadas_com_aspas
producao-mar-2005-2009.csv,60004,0
producao-mar-2010-2012.csv,37103,0
producao-mar-2013-2015.csv,38758,0
producao-mar-2016-2018.csv,56337,0
producao-mar-2019.csv,22173,15155
producao-mar-2020.csv,21815,13972
producao-mar-2021.csv,21149,12548
producao-mar-2022.csv,19225,0
producao-mar-2023.csv,14824,0
producao_por_poco_2024.csv,15229,0


##### 3.2.4.2.3 Teste de normalização

A análise confirmou que o padrão de linhas delimitadas externamente por aspas está concentrado nos arquivos de **2019, 2020 e 2021**, coincidindo com os períodos que apresentaram inconsistências durante a avaliação inicial dos atributos categóricos.

Para verificar se essa característica era efetivamente responsável pela interpretação incorreta dos registros, será realizado um teste de normalização nos três arquivos afetados.

Nas linhas que apresentam esse padrão, serão removidas as aspas externas que envolvem o registro completo e as aspas internas duplicadas serão restauradas. Em seguida, as linhas serão novamente interpretadas como registros CSV.

O procedimento será realizado apenas em memória para fins de diagnóstico, sem modificar os arquivos originais armazenados na camada RAW.

In [0]:
# Testa a correção do padrão identificado nos arquivos de 2019 a 2021.
# A normalização é realizada apenas em memória e não modifica os arquivos RAW.

anos_teste = [2019, 2020, 2021]

resultados_correcao = []

for ano in anos_teste:

    caminho = f"{caminho_raw}producao-mar-{ano}.csv"

    df_texto = (
        spark.read
        .option("encoding", "ISO-8859-1")
        .text(caminho)
    )

    # Nas linhas afetadas, remove as aspas externas do registro
    # e restaura as aspas internas duplicadas antes do parsing CSV.
    df_normalizado = (
        df_texto
        .withColumn(
            "value",
            F.when(
                F.col("value").startswith(f'"{ano}'),
                F.regexp_replace(
                    F.regexp_replace(
                        F.regexp_replace(
                            F.col("value"),
                            '^"', ''
                        ),
                        '"$', ''
                    ),
                    '""', '"'
                )
            ).otherwise(F.col("value"))
        )
    )

    # Utiliza a estrutura da respectiva tabela Bronze para testar
    # novamente a interpretação dos registros normalizados.
    schema_ddl = spark.table(
        f"workspace.bronze.producao_mar_{ano}"
    ).schema.toDDL()

    df_corrigido = (
        df_normalizado
        .filter(~F.col("value").startswith("Ano,"))
        .select(
            F.from_csv(
                F.col("value"),
                schema_ddl,
                {
                    "sep": ",",
                    "quote": '"'
                }
            ).alias("dados")
        )
        .select("dados.*")
    )

    resultado = (
        df_corrigido
        .select(
            F.lit(ano).alias("ano"),
            F.countDistinct("Estado").alias("estados_distintos"),
            F.countDistinct("Bacia").alias("bacias_distintas"),
            F.countDistinct("Campo").alias("campos_distintos"),
            F.countDistinct("Ambiente").alias("ambientes_distintos"),
            F.countDistinct("Instalação").alias("instalacoes_distintas")
        )
        .withColumn("total_registros", F.lit(df_corrigido.count()))
    )

    resultados_correcao.append(resultado)

df_resultado_correcao = resultados_correcao[0]

for resultado in resultados_correcao[1:]:
    df_resultado_correcao = df_resultado_correcao.union(resultado)

display(df_resultado_correcao)

ano,estados_distintos,bacias_distintas,campos_distintos,ambientes_distintos,instalacoes_distintas,total_registros
2019,8,9,106,1,121,22173
2020,8,9,103,1,115,21815
2021,8,9,103,1,81,21149


##### 3.2.4.2.4 Resultado do diagnóstico

Após a normalização experimental, as cardinalidades dos atributos categóricos passaram a apresentar valores compatíveis com sua natureza. Os três períodos passaram a apresentar **8 estados, 9 bacias e 1 ambiente**, além de quantidades de campos e instalações coerentes com o comportamento observado no histórico.

A quantidade de registros também foi preservada após a normalização: **22.173 registros em 2019, 21.815 em 2020 e 21.149 em 2021**, correspondendo às quantidades existentes nos respectivos arquivos de origem.

Os resultados confirmaram que as inconsistências identificadas durante a avaliação inicial não representavam características dos dados, mas eram consequência da interpretação do padrão de formatação existente em parte dos registros durante o parsing dos arquivos CSV.

Como a inconsistência era introduzida durante a ingestão, sua correção foi incorporada ao processo de construção da **camada Bronze**, conforme documentado no notebook `02_ingestao_bronze`. Os arquivos originais armazenados na camada RAW foram mantidos inalterados.

#### 3.2.4.3 Validação após a correção da ingestão

Após a incorporação da correção ao processo de ingestão da camada Bronze, a análise de cardinalidade dos atributos categóricos foi executada novamente nas 11 tabelas.

O objetivo é verificar se as inconsistências anteriormente identificadas foram eliminadas e se os dados atualmente armazenados na Bronze apresentam comportamento compatível ao longo do histórico.

In [0]:
# Reavalia as cardinalidades dos atributos categóricos após
# a correção definitiva da estratégia de ingestão na camada Bronze.

atributos_categoricos = [
    "Estado",
    "Bacia",
    "Campo",
    "Ambiente",
    "Instalação"
]

resultado_categoricos = []

for tabela in tabelas_bronze:

    df = spark.table(f"workspace.bronze.{tabela}")

    resultado = [tabela]

    for atributo in atributos_categoricos:

        # Considera os colchetes presentes nos cabeçalhos
        # dos arquivos referentes aos períodos mais recentes.
        coluna = f"[{atributo}]" if f"[{atributo}]" in df.columns else atributo

        quantidade = (
            df
            .select(coluna)
            .distinct()
            .count()
        )

        resultado.append(quantidade)

    resultado_categoricos.append(tuple(resultado))

df_categoricos = spark.createDataFrame(
    resultado_categoricos,
    [
        "tabela",
        "estados_distintos",
        "bacias_distintas",
        "campos_distintos",
        "ambientes_distintos",
        "instalacoes_distintas"
    ]
)

display(df_categoricos.orderBy("tabela"))

tabela,estados_distintos,bacias_distintas,campos_distintos,ambientes_distintos,instalacoes_distintas
producao_mar_2005_2009,9,9,80,1,117
producao_mar_2010_2012,8,9,92,1,126
producao_mar_2013_2015,8,9,101,1,165
producao_mar_2016_2018,9,9,114,1,144
producao_mar_2019,8,9,106,1,121
producao_mar_2020,8,9,103,1,115
producao_mar_2021,8,9,103,1,81
producao_mar_2022,9,9,102,1,83
producao_mar_2023,8,8,75,1,83
producao_mar_2024,7,7,73,1,81


#### 3.2.4.4 Resultado da análise de consistência

A reexecução da análise após a correção da ingestão demonstrou que as cardinalidades anômalas anteriormente observadas nos períodos de 2019, 2020 e 2021 não estão mais presentes.

Os atributos `Estado` e `Bacia` passaram a apresentar baixa variação de cardinalidade entre os períodos, enquanto `Campo` e `Instalação` apresentam variações compatíveis com a evolução dos registros ao longo do histórico. Essas diferenças não são interpretadas automaticamente como inconsistências, uma vez que campos e instalações podem surgir ou deixar de apresentar produção ao longo do período analisado.

O atributo `Ambiente` apresenta apenas um valor distinto em todas as 11 tabelas, comportamento coerente com o recorte marítimo utilizado neste projeto.

A análise de qualidade permitiu, portanto, identificar uma inconsistência originada durante o parsing dos arquivos de 2019 a 2021, diagnosticar sua causa nos arquivos de origem e validar a correção posteriormente incorporada à camada Bronze.

Com a correção da ingestão, os atributos categóricos analisados apresentam comportamento consistente para prosseguir com as etapas de padronização e consolidação da camada Silver.

### 3.2.5 Validade dos atributos quantitativos

A dimensão de validade tem como objetivo verificar se os valores armazenados são compatíveis com a natureza dos atributos analisados.

Nesta etapa serão avaliados os atributos quantitativos relacionados à produção e à injeção de fluidos, buscando identificar principalmente a ocorrência de valores negativos.

Como os tipos de dados ainda não estão padronizados entre os diferentes períodos históricos e parte dos valores numéricos está armazenada como `string` utilizando a convenção brasileira de separadores decimais, será realizada uma conversão temporária desses atributos exclusivamente para fins de análise.

Nenhuma alteração será realizada nas tabelas Bronze. A conversão definitiva dos atributos quantitativos será implementada posteriormente durante a construção da camada Silver.

#### 3.2.5.1 Preparação dos atributos quantitativos

Serão avaliados os atributos relacionados aos volumes de produção e injeção presentes nos arquivos da ANP.

Como esses atributos apresentam diferentes tipos de dados entre as tabelas Bronze, a conversão temporária utilizada nesta análise considerará o tipo originalmente identificado em cada período. Valores já armazenados em tipos numéricos serão apenas convertidos para `double`, enquanto valores armazenados como `string` serão interpretados considerando a convenção numérica brasileira presente nos arquivos de origem.

Essa conversão será utilizada exclusivamente para a avaliação de qualidade e não modifica os dados persistidos na camada Bronze.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

atributos_quantitativos = [
    "Produção de Óleo (m³)",
    "Produção de Condensado (m³)",
    "Produção de Gás Associado (Mm³)",
    "Produção de Gás Não Associado (Mm³)",
    "Produção de Água (m³)",
    "Injeção de Gás (Mm³)",
    "Injeção de Água para Recuperação Secundária (m³)",
    "Injeção de Água para Descarte (m³)",
    "Injeção de Gás Carbônico (Mm³)",
    "Injeção de Nitrogênio (Mm³)",
    "Injeção de Vapor de Água (t)",
    "Injeção de Polímeros (m³)",
    "Injeção de Outros Fluidos (m³)"
]

# Converte temporariamente os atributos quantitativos para double,
# respeitando o tipo originalmente armazenado na camada Bronze.
def converter_numero_brasileiro(df, coluna):
    tipo_coluna = df.schema[coluna].dataType
    coluna_spark = F.col(f"`{coluna}`")

    if isinstance(tipo_coluna, StringType):
        return (
            F.regexp_replace(
                F.regexp_replace(
                    coluna_spark,
                    r"\.",
                    ""
                ),
                ",",
                "."
            )
            .cast("double")
        )

    return coluna_spark.cast("double")

#### 3.2.5.2 Identificação de valores negativos

Após a conversão temporária dos atributos quantitativos, será contabilizada a ocorrência de valores negativos em cada variável e período histórico.

Como os atributos representam volumes de produção ou injeção, valores inferiores a zero serão destacados para investigação. Sua ocorrência não será automaticamente corrigida ou excluída nesta etapa, pois qualquer tratamento dependerá da compreensão de seu significado na fonte.

In [0]:
# Verifica a ocorrência de valores negativos nos atributos quantitativos
# em todas as tabelas Bronze, sem modificar os dados persistidos.

resultado_negativos = []

for tabela in tabelas_bronze:

    df = spark.table(f"workspace.bronze.{tabela}")

    for atributo in atributos_quantitativos:

        # Considera os colchetes presentes nos cabeçalhos
        # dos arquivos referentes aos períodos mais recentes.
        coluna = f"[{atributo}]" if f"[{atributo}]" in df.columns else atributo

        df_validacao = df.select(
            converter_numero_brasileiro(df, coluna).alias("valor")
        )

        quantidade_negativos = (
            df_validacao
            .filter(F.col("valor") < 0)
            .count()
        )

        resultado_negativos.append(
            (
                tabela,
                atributo,
                quantidade_negativos
            )
        )

df_negativos = spark.createDataFrame(
    resultado_negativos,
    [
        "tabela",
        "atributo",
        "valores_negativos"
    ]
)

display(
    df_negativos
    .filter(F.col("valores_negativos") > 0)
    .orderBy("tabela", "atributo")
)

tabela,atributo,valores_negativos
producao_mar_2013_2015,Produção de Água (m³),3


#### 3.2.5.3 Investigação dos valores negativos

A análise identificou **três valores negativos** no atributo `Produção de Água (m³)`, todos localizados na tabela referente ao período de 2013 a 2015.

Como valores negativos podem representar tanto uma inconsistência quanto uma característica específica do dado registrado na fonte, esses registros serão analisados individualmente antes da definição de qualquer regra de tratamento.

Para isso, serão recuperados os registros correspondentes diretamente da camada Bronze, preservando seus valores originais e os principais atributos utilizados para sua identificação.

In [0]:
# Recupera os registros que apresentaram produção de água negativa
# para permitir sua investigação antes de definir qualquer tratamento.

df_2013_2015 = spark.table(
    "workspace.bronze.producao_mar_2013_2015"
)

df_agua_negativa = (
    df_2013_2015
    .withColumn(
        "_producao_agua_numerica",
        converter_numero_brasileiro("`Produção de Água (m³)`")
    )
    .filter(F.col("_producao_agua_numerica") < 0)
    .select(
        "Ano",
        "Mês/Ano",
        "Estado",
        "Bacia",
        "Campo",
        "Poço",
        "Instalação",
        "Produção de Água (m³)",
        "_producao_agua_numerica"
    )
)

display(df_agua_negativa)

Ano,Mês/Ano,Estado,Bacia,Campo,Poço,Instalação,Produção de Água (m³),_producao_agua_numerica
2013,11/2013,Espírito Santo,Campos,ARGONAUTA,7-ARGO-14H-ESS,FPSO ESPIRITO SANTO,"-0,0268",-0.0268
2013,11/2013,Espírito Santo,Campos,ARGONAUTA,7-ARGO-5H-ESS,FPSO ESPIRITO SANTO,"-0,01539",-0.01539
2013,11/2013,Espírito Santo,Campos,ARGONAUTA,7-ARGO-9H-ESS,FPSO ESPIRITO SANTO,"-0,00908",-0.00908


#### 3.2.5.4 Verificação nos dados de origem

A investigação dos registros demonstrou que os três valores negativos já estão presentes na coluna original `Produção de Água (m³)` da tabela Bronze e não foram introduzidos pela conversão temporária utilizada na análise de validade.

Entretanto, para verificar se esses valores também estão presentes nos dados fornecidos pela fonte ou se poderiam ter sido introduzidos durante a ingestão, os registros correspondentes serão consultados diretamente no arquivo original armazenado na camada RAW.

Essa verificação permitirá determinar se os valores negativos foram preservados desde a fonte ou se existe alguma inconsistência adicional no processo de ingestão.

In [0]:
# Recupera diretamente no arquivo RAW apenas os registros correspondentes
# aos valores negativos identificados anteriormente na camada Bronze.

caminho_2013_2015 = (
    "/Volumes/workspace/default/anp_raw/producao-mar-2013-2015.csv"
)

# Obtém automaticamente os poços identificados na análise anterior,
# evitando a transcrição manual de seus identificadores.
pocos_negativos = [
    linha["Poço"]
    for linha in df_agua_negativa.select("Poço").distinct().collect()
]

df_raw_texto = spark.read.text(caminho_2013_2015)

# Mantém apenas as linhas de 11/2013 referentes aos poços
# que apresentaram produção de água negativa na Bronze.
condicao_pocos = F.lit(False)

for poco in pocos_negativos:
    condicao_pocos = condicao_pocos | F.col("value").contains(poco)

df_raw_negativos = (
    df_raw_texto
    .filter(F.col("value").startswith("2013,11/2013,"))
    .filter(condicao_pocos)
)

display(df_raw_negativos)

value
"2013,11/2013,Espírito Santo,Campos,ARGONAUTA,7-ARGO-14H-ESS,Mar,FPSO ESPIRITO SANTO,""3938,89727"",0,""230,81677"",0,""-0,0268"",,,,,,,,"
"2013,11/2013,Espírito Santo,Campos,ARGONAUTA,7-ARGO-5H-ESS,Mar,FPSO ESPIRITO SANTO,""5275,09795"",0,""310,98604"",0,""-0,01539"",,,,,,,,"
"2013,11/2013,Espírito Santo,Campos,ARGONAUTA,7-ARGO-9H-ESS,Mar,FPSO ESPIRITO SANTO,""5460,36048"",0,""350,91553"",0,""-0,00908"",,,,,,,,"


#### 3.2.5.5 Resultado da análise de validade

A avaliação dos atributos quantitativos identificou apenas **três valores negativos** em todo o histórico analisado, todos no atributo `Produção de Água (m³)` e concentrados em registros referentes a **novembro de 2013**.

A investigação individual dos registros confirmou que os valores negativos já estavam presentes na coluna original da camada Bronze. Em seguida, a consulta direta ao arquivo `producao-mar-2013-2015.csv`, preservado na camada RAW, confirmou que os mesmos valores constam nos dados de origem.

Dessa forma, os valores negativos **não foram introduzidos pelo processo de ingestão nem pela conversão temporária utilizada nesta análise**.

Como não há evidência suficiente para caracterizá-los como erro da fonte, os três registros serão preservados durante a construção da camada Silver. Não serão aplicadas substituições por zero, remoções ou outras correções arbitrárias.

A análise também não identificou valores negativos nos demais atributos quantitativos avaliados.

Com isso, a avaliação de validade não indica necessidade de exclusão ou correção de registros antes da construção da camada Silver.

## 3.3 Síntese da avaliação de qualidade

A avaliação de qualidade permitiu identificar as principais características e inconsistências dos dados antes de sua consolidação na camada Silver.

A tabela a seguir sintetiza os resultados obtidos nas dimensões de estrutura, completude, unicidade, consistência e validade.

| Dimensão | Achado | Impacto para a Silver |
|---|---|---|
| **Estrutura** | As 11 tabelas possuem a mesma estrutura lógica de 21 atributos, porém existem diferenças nos nomes e tipos das colunas entre os períodos. | Os nomes e tipos deverão ser padronizados antes da consolidação. |
| **Estrutura** | Os cabeçalhos dos arquivos de 2022 a 2025 apresentam colchetes. | Os colchetes serão removidos durante a padronização dos nomes das colunas. |
| **Tipos de dados** | Sete atributos apresentam alternância entre `integer` e `string` ao longo do histórico. | Os tipos serão definidos explicitamente na Silver, evitando dependência da inferência automática. |
| **Completude** | Foram identificados diferentes padrões de valores nulos, principalmente nos atributos de produção e injeção. | Os valores nulos serão preservados, evitando sua substituição automática por zero. |
| **Unicidade** | Não foram identificadas linhas completamente duplicadas nas 11 tabelas Bronze. | Nenhum registro será removido por duplicidade. |
| **Granularidade** | A combinação `Mês/Ano + Poço + Campo + Instalação` não apresentou repetições no histórico analisado. | Essa combinação representa a granularidade observada nos dados e será preservada na consolidação. |
| **Consistência** | A análise inicial identificou problemas de parsing nos arquivos de 2019 a 2021 devido a um padrão específico de aspas. | A causa foi corrigida na etapa de ingestão da Bronze; nenhuma correção adicional será necessária na Silver para esse problema. |
| **Validade** | Foram identificados três valores negativos em `Produção de Água (m³)`, todos presentes também no arquivo RAW de origem. | Os valores serão preservados, pois não há evidência suficiente para classificá-los como erro da fonte. |

Os resultados demonstram que a construção da camada Silver deverá priorizar a **padronização estrutural e de tipos**, preservando os dados de origem sempre que não houver evidência suficiente para justificar sua alteração.

A avaliação de qualidade também permitiu evitar tratamentos arbitrários, como substituição indiscriminada de valores nulos, remoção de registros ou correção de valores apenas por apresentarem comportamento incomum.

## 3.4 Regras de tratamento definidas para a camada Silver

Com base nos resultados da avaliação de qualidade, foram definidas as regras que orientarão a transformação e consolidação dos dados na camada Silver.

As transformações serão aplicadas somente quando necessárias para garantir consistência estrutural, compatibilidade entre os períodos históricos e utilização analítica dos dados.

As seguintes regras foram estabelecidas:

1. **Padronizar os nomes das colunas**, removendo os colchetes existentes nos cabeçalhos dos períodos de 2022 a 2025 e adotando uma nomenclatura única para todo o histórico.

2. **Definir explicitamente os tipos de dados**, evitando a dependência da inferência automática utilizada durante a ingestão.

3. **Converter os atributos quantitativos para tipos numéricos**, considerando a convenção de representação existente nos arquivos de origem.

4. **Converter o atributo de período para um tipo adequado de data**, permitindo posteriormente a construção da dimensão temporal e análises cronológicas.

5. **Preservar os valores nulos**, não realizando substituição automática por zero.

6. **Preservar todos os registros existentes**, uma vez que não foram identificadas duplicidades na granularidade observada.

7. **Preservar os três valores negativos identificados em `Produção de Água (m³)`**, pois a investigação confirmou sua presença nos arquivos originais da fonte.

8. **Consolidar as 11 tabelas Bronze em uma única tabela histórica**, abrangendo o período de 2005 a 2025.

9. **Validar a tabela consolidada após as transformações**, verificando quantidade de registros, período coberto, estrutura, tipos de dados e possíveis perdas decorrentes das conversões.

## 3.5 Conclusão da avaliação da qualidade dos dados

A avaliação de qualidade permitiu compreender as principais diferenças existentes entre os períodos históricos e definir as regras necessárias para sua integração.

Além de identificar diferenças estruturais e de tipos, a análise permitiu detectar uma inconsistência de parsing nos arquivos de 2019 a 2021, cuja causa foi investigada nos dados de origem e corrigida na etapa de ingestão da camada Bronze.

Também foram avaliados valores nulos, duplicidades, granularidade, consistência dos atributos categóricos e validade dos atributos quantitativos, permitindo que as decisões de tratamento fossem fundamentadas nas características efetivamente observadas nos dados.

Com as regras de tratamento definidas, a próxima etapa do pipeline consiste na construção da camada Silver, responsável pela padronização e consolidação do histórico de produção offshore entre 2005 e 2025.